In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
import os
import tensorflow as tf
from datetime import datetime

from utils.data_loader import create_datasets
from utils.models import build_baseline_cnn

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
model_name = 'RAF'

Load pretrained model to fine-tune it? If yes, specify model name, if no set it to `None`.

In [ ]:
pretrained_model_name = None

# Data

## Preprocessing Function
Pixel values per channel are normalized to have zero mean and unit variance.

In [ ]:
# RAF-DB training data mean and std
mean = [146.6770, 114.6274, 102.3102]
std = [67.6282, 61.7651, 61.3665]

def preprocess(x):
    x = np.array(x, dtype='float32')
    x[..., 0] -= mean[0]
    x[..., 1] -= mean[1]
    x[..., 2] -= mean[2]
    if std is not None:
        x[..., 0] /= std[0]
        x[..., 1] /= std[1]
        x[..., 2] /= std[2] 
    return x

def de_preprocess(x):
    x = np.array(x, dtype='float32')
    if std is not None:
        x[..., 0] *= std[0]
        x[..., 1] *= std[1]
        x[..., 2] *= std[2]
    x[..., 0] += mean[0]
    x[..., 1] += mean[1]
    x[..., 2] += mean[2]
    return np.clip(x, 0, 255).astype('uint8')

## Load Data

In [ ]:
%%time

IMG_SIZE = (100, 100)
IMG_SHAPE = IMG_SIZE + (3,)
BATCH_SIZE = 16

train_ds, val_ds, test_ds, class_weights, info = create_datasets(
    data_dir='data/DATASET',
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    val_split=0.2,
    augment_train=True
)

In [ ]:
emotion_labels = ['surprise', 'fear', 'disgust', 'happiness', 'sadness', 'anger', 'neutral']
num_classes = len(emotion_labels)
print('Number of classes: ', num_classes)

## Data Examples

In [ ]:
images, labels = next(iter(train_ds))
images_np = images.numpy()
labels_np = labels.numpy()

plt.subplots(4, 4, figsize=(10, 10))
for i in range(min(16, len(images_np))):
    plt.subplot(4, 4, i+1)
    plt.imshow(de_preprocess(images_np[i]))
    plt.title(emotion_labels[int(labels_np[i])])
    plt.axis('off')
plt.suptitle('Training Data Examples', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Model

## Build Model
Model đã bao gồm softmax layer — KHÔNG cần thêm Dense layer.

In [ ]:
model = build_baseline_cnn(input_shape=IMG_SHAPE, num_classes=num_classes, dropout_rate=0.25)

if pretrained_model_name is not None:
    model.load_weights(f'./models/{pretrained_model_name}')

# sparse_categorical_crossentropy vi labels la integer (0-6)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Training

In [ ]:
dt = datetime.now().strftime('%m%d-%H%M')
os.makedirs('./modelcheckpoints', exist_ok=True)
os.makedirs('./log', exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=f'./modelcheckpoints/{model_name}_{dt}.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.1,
        patience=4,
        verbose=1,
        mode='auto',
        min_lr=1e-8
    ),
    tf.keras.callbacks.CSVLogger(
        f'./log/{model_name}_{dt}.csv',
        separator=',', append=True
    )
]

In [ ]:
%%time

epochs = 20

history = model.fit(
    train_ds,
    epochs=epochs,
    validation_data=val_ds,
    class_weight=class_weights,
    callbacks=callbacks
)

# Results

## Training Curves

In [ ]:
plt.figure(figsize=(14, 5))

if 'history' in locals():
    history_dict = history.history
else:
    log_files = glob.glob('log/RAF_*.csv')
    if log_files:
        latest_log = max(log_files, key=os.path.getmtime)
        history_dict = pd.read_csv(latest_log).to_dict(orient='list')
    else:
        history_dict = None
        print('No history found.')

if history_dict:
    ep = range(1, len(history_dict['accuracy']) + 1)
    plt.subplot(1,2,1)
    plt.plot(ep, history_dict['accuracy'], 'b-o', ms=3, lw=2, label='Training')
    plt.plot(ep, history_dict['val_accuracy'], 'r-o', ms=3, lw=2, label='Validation')
    plt.title('Accuracy', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy')
    plt.grid(True, alpha=0.3); plt.legend()
    
    plt.subplot(1,2,2)
    plt.plot(ep, history_dict['loss'], 'b-o', ms=3, lw=2, label='Training')
    plt.plot(ep, history_dict['val_loss'], 'r-o', ms=3, lw=2, label='Validation')
    plt.title('Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.grid(True, alpha=0.3); plt.legend()
    
    plt.suptitle('Training History', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f'Best Val Accuracy: {max(history_dict["val_accuracy"]):.4f}')

## Test Results

In [ ]:
loss, acc = model.evaluate(test_ds, verbose=2)
print(f'\nAccuracy:\t{acc * 100:.2f}%')
print(f'Loss:\t\t{loss:.4f}')

## Confusion Matrix & Classification Report

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print('Classification Report:')
print('=' * 60)
print(classification_report(y_true, y_pred, target_names=emotion_labels, digits=4))

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=emotion_labels, yticklabels=emotion_labels,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (Normalized)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=emotion_labels, yticklabels=emotion_labels,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Confusion Matrix (Counts)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout()
plt.show()

## Per-class Accuracy

In [ ]:
per_class_acc = cm_norm.diagonal()
colors = ['#FF6B6B','#845EC2','#4B8B3B','#FFD93D','#4A90D9','#FF4444','#95A5A6']
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(emotion_labels, per_class_acc, color=colors, edgecolor='black', lw=0.5)
for bar, v in zip(bars, per_class_acc):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.axhline(y=np.mean(per_class_acc), color='red', ls='--', alpha=0.7,
           label=f'Mean: {np.mean(per_class_acc):.1%}')
ax.set_ylabel('Accuracy'); ax.set_title('Per-class Accuracy', fontsize=16, fontweight='bold')
ax.set_ylim(0,1.15); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

# Save Model

In [ ]:
os.makedirs('outputs/models', exist_ok=True)
model.save('outputs/models/baseline_cnn.keras')
print('Da luu model thanh cong!')
print('Path: outputs/models/baseline_cnn.keras')
print('Chay Webcam demo: python app/run.py')

# Webcam Demo
Chay cell duoi de mo webcam. Nhan **q** de thoat.

In [ ]:
import cv2
from utils.data_loader import preprocess_raf_np

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
COLORS = {'surprise':(0,215,255),'fear':(130,92,194),'disgust':(59,139,75),
          'happiness':(0,215,255),'sadness':(217,144,74),'anger':(60,76,231),'neutral':(166,165,149)}

cap = cv2.VideoCapture(0)
print('Webcam opened. Press Q to quit.')

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(48,48))
    for (x,y,w,h) in faces:
        face_rgb = cv2.cvtColor(frame[y:y+h, x:x+w], cv2.COLOR_BGR2RGB)
        face_input = np.expand_dims(preprocess_raf_np(cv2.resize(face_rgb, IMG_SIZE)), 0)
        preds = model.predict(face_input, verbose=0)[0]
        eidx = np.argmax(preds); emo = emotion_labels[eidx]; conf = preds[eidx]
        color = COLORS.get(emo, (255,255,255))
        cv2.rectangle(frame, (x,y), (x+w,y+h), color, 2)
        cv2.putText(frame, f'{emo}: {conf:.1%}', (x,y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
    cv2.imshow('Emotion Recognition - Press Q to quit', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break
cap.release(); cv2.destroyAllWindows()
print('Webcam closed.')

---